# 03 실전 반도체 공정 데이터 분석 · 복습 빈칸 채우기 — 교사용 (정답)

학생용 `복습_빈칸채우기_학생용.ipynb`의 정답이 채워진 파일입니다. 위에서 아래로 그대로 실행됩니다.

- 각 문제 아래의 **정답**에 학생용 `___` 자리에 들어갈 값을 순서대로 적어 두었습니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.

### 준비 · 라이브러리 불러오기

빈칸이 없는 셀입니다. 먼저 실행하세요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             accuracy_score, recall_score)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1~2단계 · 데이터 불러오기 & 탐색

컬럼이 590개가 넘으므로 요약 정보로 살핍니다.

### Q1. 데이터 읽고 크기 확인하기

fab.csv를 읽고 행·열 수를 출력합니다.

**정답:** `read_csv` · `shape`

In [ ]:
df = pd.read_csv('fab.csv')
print(f"데이터 크기: {df.shape[0]}행 × {df.shape[1]}열")

### Q2. 열 목록 없이 요약 보기

열이 너무 많으니 열별 목록은 빼고 요약만 봅니다.

**정답:** `verbose`

In [ ]:
df.info(verbose=False)

### Q3. 라벨 분포 확인하기

정상(-1)과 불량(+1)이 각각 몇 건인지 봅니다. 불균형이 얼마나 심한지 확인하세요.

**정답:** `value_counts`

In [ ]:
df['Pass_Fail'].value_counts()

## 3~4단계 · 결측값 처리 · 의미 없는 컬럼 제거

### Q4. 컬럼별 결측률 계산하기

결측값 개수를 전체 행 수로 나눠 %로 만듭니다.

**정답:** `isnull` · `len`

In [ ]:
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head())

### Q5. 결측률 50% 초과 컬럼 제거하기

결측이 너무 많은 컬럼을 통째로 지웁니다.

**정답:** `>` · `drop`

In [ ]:
cols_to_drop = miss_pct[miss_pct > 50].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f"제거한 컬럼: {len(cols_to_drop)}개, 남은 컬럼: {df.shape[1]}개")

### Q6. 남은 결측값을 중앙값으로 채우기

타겟(Pass_Fail)은 목록에서 빼고, 센서 열만 중앙값으로 채웁니다.

**정답:** `remove` · `median`

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # 타겟은 채우지 않습니다

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"남은 결측: {df.isnull().sum().sum()}")

### Q7. 분산이 0인 컬럼 제거하기

값이 변하지 않는(분산 0 또는 거의 0) 센서를 지웁니다.

**정답:** `var` · `==`

In [ ]:
variances = df[numeric_cols].var()
constant_cols = variances[variances == 0].index.tolist()
near_constant = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]
print(f"사용 가능한 센서: {len(numeric_cols)}개")

## 5~6단계 · 특성 선택 · 핵심 변수 EDA

### Q8. 상위 20개 센서 자동 선택하기

불량(+1)을 1로 바꾼 y를 만들고, ANOVA F-검정으로 상위 20개 센서를 고릅니다.

**정답:** `1` · `SelectKBest` · `f_classif` · `scores_` · `False`

In [ ]:
y = (df['Pass_Fail'] == 1).astype(int)

selector = SelectKBest(score_func=f_classif, k=20)
selector.fit(df[numeric_cols], y)

f_scores = pd.Series(selector.scores_, index=numeric_cols).replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(20).index.tolist()
print("상위 5개 센서:", top_k_cols[:5])

### Q9. 1위 센서의 정상/불량 분포 비교하기

F-점수 1위 센서를 박스플롯으로 그려 봅니다.

**정답:** `boxplot`

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(x='Pass_Fail', y=top_k_cols[0], data=df,
            hue='Pass_Fail', palette={-1: '#2ecc71', 1: '#e74c3c'}, legend=False)
plt.title(f'{top_k_cols[0]} — 정상(-1) vs 불량(+1)')
plt.show()

## 7~9단계 · 전처리 · 모델 학습 · 평가

### Q10. 학습/테스트 데이터 나누기

20%를 테스트로 떼어 내고, 불량 비율이 양쪽에 똑같이 유지되게 나눕니다.

**정답:** `train_test_split` · `0.2` · `stratify`

In [ ]:
X = df[top_k_cols]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"학습: {X_train.shape}, 불량 {y_train.sum()}건 / 테스트: {X_test.shape}, 불량 {y_test.sum()}건")

### Q11. 표준화하기

학습 데이터로만 기준(평균·표준편차)을 정하고, 테스트 데이터는 그 기준으로 바꾸기만 합니다.

**정답:** `fit_transform` · `transform`

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Q12. 로지스틱 회귀 학습하기

적은 쪽(불량)에 가중치를 주고 학습합니다.

**정답:** `'balanced'` · `fit`

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

### Q13. 랜덤 포레스트 학습하기

나무(tree) 200개로 이루어진 숲(forest) 모델을 학습합니다.

**정답:** `RandomForestClassifier`

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8,
                                  random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

### Q14. 혼동행렬과 분류 리포트 보기

로지스틱 회귀의 예측을 혼동행렬과 리포트로 평가합니다.

**정답:** `predict` · `confusion_matrix`

In [ ]:
y_pred_lr = lr_model.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=['정상(0)', '불량(1)']))

### Q15. ROC 곡선의 AUC 구하기

불량일 확률로 ROC 곡선을 만들고 곡선 아래 면적(AUC)을 구합니다.

**정답:** `predict_proba` · `roc_curve` · `auc`

In [ ]:
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
print(f"로지스틱 회귀 AUC: {auc(fpr, tpr):.3f}")

### Q16. 특성 중요도 보기

랜덤 포레스트가 중요하게 쓴 센서 상위 5개를 봅니다.

**정답:** `feature_importances_`

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=top_k_cols)
print(importance.sort_values(ascending=False).head().round(3))

## 10 · 보강 실습

### Q17. '항상 정상' 기준 모델과 비교하기

모두 정상(0)이라고 예측하면 정확도와 불량 Recall이 어떻게 되는지 봅니다.

**정답:** `zeros`

In [ ]:
always_normal = np.zeros(len(y_test), dtype=int)
print(f"정확도: {accuracy_score(y_test, always_normal):.3f}")
print(f"불량 Recall: {recall_score(y_test, always_normal, zero_division=0):.3f}")

### Q18. 혼동행렬 숫자로 Recall 직접 계산하기

혼동행렬을 네 값으로 나누고 불량 Recall = TP / (TP + FN)을 계산합니다.

**정답:** `ravel` · `fn`

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr).ravel()
manual_recall = tp / (tp + fn)
print(f"TN {tn}, FP {fp}, FN {fn}, TP {tp}")
print(f"불량 Recall: {manual_recall:.3f}")

### Q19. 임계값을 낮춰 보기

불량 확률이 0.3 이상이면 불량으로 판정했을 때 Recall을 봅니다.

**정답:** `>=`

In [ ]:
pred_03 = (y_prob_lr >= 0.3).astype(int)
print(f"임계값 0.3 불량 Recall: {recall_score(y_test, pred_03):.3f}")
print(f"임계값 0.5 불량 Recall: {recall_score(y_test, y_pred_lr):.3f}")

## 마무리

학생용 파일은 이 파일에서 정답 자리만 `___`로 바꾼 것입니다.